<style>
.jp-Notebook, .notebook { max-width: 1000px; margin: 0 auto; }

div.text_cell_render {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Arial, sans-serif;
    line-height: 1.6;
    color: #333;
}

h1, h2, h3, h4, h5, h6 {
    margin-top: 1.5em;
    margin-bottom: 0.5em;
    font-weight: 600;
    color: #2c6e9e;
}

h1 { font-size: 2em; border-bottom: 2px solid #2c6e9e; padding-bottom: 10px; }
h2 { font-size: 1.5em; border-bottom: 1px solid #ddd; padding-bottom: 5px; }
h3 { font-size: 1.3em; }

p, li { text-align: justify; }

.atencao, .box-warning {
    background-color: #fff3cd;
    border-left: 5px solid #ff9800;
    padding: 15px;
    border-radius: 8px;
    margin: 15px 0;
}
.info, .box-info {
    background-color: #e8f4fd;
    border-left: 5px solid #2196f3;
    padding: 15px;
    border-radius: 8px;
    margin: 15px 0;
}
.sucesso, .box-success {
    background-color: #e6f4ea;
    border-left: 5px solid #4caf50;
    padding: 15px;
    border-radius: 8px;
    margin: 15px 0;
}

table { border-collapse: collapse; width: 100%; margin: 15px 0; }
th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
th { background-color: #f2f2f2; font-weight: 600; }
</style>

<div style="max-width: 900px; margin: 0 auto; font-family: Arial, sans-serif; line-height: 1.6;">

<br>

<div align="center">
    <img src="
X9u0k/YOdr2lLWiu3/+ezLoWd1WMyO7FPDc4dtodGvMGwWMsRsbWmlFLWkPXR0XngnaOpr2fJe3xYonRFxTH/wHIBpxd2aZzVAAAAABJRU5ErkJggg" width="150"/>
</div>

<br>

<div align="center">
    <h1 style="font-size: 30px; font-weight: 900; margin: 0; padding: 0;">Projeto GEOTEC</h1>
</div>

---

<div align="center">
    <p style="font-size: 18px; margin-top: 10px; margin-bottom: 10px;">
        Capacitação em Geotecnologias para Monitoramento do Crédito Rural e do Proagro
    </p>
</div>

<div align="center">
    <p><b>Módulo:</b> Governança Territorial</p>
</div>

---

### Equipe

Pedro Coutinho, Gerd Sparovek, Alberto Barretto, Herbert Lincon, Marluce Scarabello, Pietro Gragnolati, Ana Beatriz Ferreira Macedo, Rodrigo Fernando Maule, João Marinho, João Gabriel Souza, Victor Hugo Bitu Patricio, Richard Torsiano

---

### Instituição
Instituto para Governança Territorial e Políticas Públicas (iGPP)

---

</div>

# Aula 13 (parte 2) — Regularização Fundiária na Amazônia Legal
<hr style="border:1px solid #2c6e9e;">

Este notebook avalia a possibilidade de regularização fundiária de um imóvel rural (CAR) cuja categoria fundiária predominante seja **Gleba Pública (GP)** ou **Floresta Pública do Tipo B (GPFPND)**, conforme a **Lei nº 11.952/2009**.

A análise considera:
1. **Categoria fundiária predominante** do imóvel (Malha Fundiária CDT 2025)
2. **Localização em relação à Amazônia Legal** (>50% da área)
3. **Tipificação da ocupação** em pré ou pós 22/07/2008 (PRODES — INPE)
4. **Tamanho do imóvel** (limite de 2.500 ha para regularização)

Este notebook é **complementar** ao `02_indicadores_fundiarios.ipynb` (CMN nº 5.268). Ambos podem ser executados de forma independente.


<hr style="border:1px solid #2c6e9e;">

## 1. Pré-requisitos e Bibliotecas Utilizadas

### 1.1 Pré-requisitos Técnicos

<div class="atencao">
<b>ATENÇÃO:</b> Certifique-se de que todos os pré-requisitos abaixo estão atendidos antes de executar este notebook.
</div>

|| Pré-requisito | Detalhes |
|---|--------------|----------|
| 1 | **PostgreSQL 14+ com PostGIS 3+** rodando localmente | Banco `geotec` acessível em `localhost:5432` |
| 2 | **SRID 97823 cadastrado** no banco | Cônica Equivalente de Albers — Brasil |
| 3 | **Dados importados** via `01_import_dados.ipynb` | Inclusive `inpe.pa_mt_desmatamento_inpe_20260304` e `ibge.pa_br_amazonialegal_ibge_2022` |
| 4 | **Python 3.10+** com ambiente virtual ativo | Recomendado: `conda` ou `venv` |

### 1.2 Dados Necessários no Banco

| Schema | Tabela | Conteúdo | Fonte |
|--------|--------|----------|-------|
| `car` | `es_mt_car_20260406` | Cadastro Ambiental Rural — Mato Grosso | SICAR |
| `malha` | `pa_mt_malhafundiaria_2025_cdt` | Malha Fundiária CDT 2025 | INCRA / CDT |
| `incra` | `pa_br_modulosfiscais_incra` | Módulos Fiscais por município | INCRA |
| `inpe` | `pa_mt_desmatamento_inpe_20260304` | Desmatamento PRODES (MT) | INPE |
| `ibge` | `pa_br_amazonialegal_ibge_2022` | Limite Amazônia Legal | IBGE |
| `incra` | `pa_br_sigef_privado_incra_2026` | Privados SIGEF (mapa) | INCRA |
| `incra` | `pa_br_snci_privado_incra_2026` | Privados SNCI (mapa) | INCRA |


In [1]:
# Verificação do ambiente — rode esta célula antes de continuar
import sys, importlib

LIBS = [
    "pandas", "geopandas", "sqlalchemy", "psycopg2",
    "folium", "matplotlib", "shapely",
    "IPython",
]

print(f"Python: {sys.version.split()[0]}")
print(f"Ambiente: {sys.prefix}\n")

faltando = []
for lib in LIBS:
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, "__version__", "ok")
        print(f"  ✓ {lib:<18} {ver}")
    except ImportError:
        print(f"  ✗ {lib:<18} NAO INSTALADO")
        faltando.append(lib)

if faltando:
    print(f"\n[AVISO] Bibliotecas faltando: {', '.join(faltando)}")
else:
    print("\n✓ Ambiente OK — pode continuar.")


Python: 3.12.13
Ambiente: C:\Users\vhbit\miniforge3\envs\geotec

  ✓ pandas             3.0.2
  ✓ geopandas          1.1.3
  ✓ sqlalchemy         2.0.49
  ✓ psycopg2           2.9.11 (dt dec pq3 ext lo64)
  ✓ folium             0.20.0
  ✓ matplotlib         3.10.8
  ✓ shapely            2.1.2
  ✓ IPython            9.12.0

✓ Ambiente OK — pode continuar.


In [2]:
import os
import urllib.parse
from datetime import date

import pandas as pd
import geopandas as gpd
import folium

from sqlalchemy import create_engine, text
from IPython.display import HTML, display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
print("[OK] Bibliotecas importadas")


[OK] Bibliotecas importadas


<hr style="border:1px solid #2c6e9e;">

## 2. Conexão com o Banco e Parâmetros da Análise

<div class="atencao">
<b>ATENÇÃO:</b> Altere o <code>COD_IMOVEL</code> para o imóvel que deseja analisar.
Os imóveis de estudo de caso desta aula estão listados como comentário.
</div>


In [3]:
# ============================================================================
# IMOVEL DE INTERESSE - ALTERE ESTE VALOR
# ============================================================================
COD_IMOVEL = "MT-5106265-62DB7655979E41A5BB9FA3A0CD2AF9BD"  # >15MF + GPFPND

# Outros imoveis de estudo de caso:
# COD_IMOVEL = "MT-5106265-BD93832E1A6E4AAC88BED0095F569863"  # 100% GPFPND
# COD_IMOVEL = "MT-5106240-2C569B7AC0C34055986020860544229E"  # UCPI
# COD_IMOVEL = "MT-5103254-198BD8BDC80240CCA17196EC2558E146"  # UCUS
# COD_IMOVEL = "MT-5102504-5785523723BB4AF7B87D7325A4563CB8"  # AM
# COD_IMOVEL = "MT-5107743-C61355B0E15B4A47B37774C8E0BBD951"  # TINH
# COD_IMOVEL = "MT-5106109-9F3831A8EBB949D0AE0E3F29615DA2C0"  # TQT
# COD_IMOVEL = "MT-5105580-9C4D98486B8D46AFAA2D4D07F16A878F"  # TIH

# ============================================================================
# CONEXAO - Banco local geotec
# ============================================================================
DB_USER     = "postgres"
DB_PASSWORD = "postgres"
DB_HOST     = "localhost"
DB_PORT     = "5432"
DB_NAME     = "geotec"

# ============================================================================
# SCHEMAS E TABELAS
# ============================================================================

# CAR - Mato Grosso
CAR_SCHEMA = "car"
CAR_TABLE  = "es_mt_car_20260406"
CAR_COD    = "cod_imovel"
CAR_GEOM   = "geometry"
CAR_MF     = "mod_fiscal"

# Malha Fundiaria CDT (categorias predominantes)
MALHA_SCHEMA = "malha"
MALHA_TABLE  = "pa_mt_malhafundiaria_2025_cdt"
MALHA_GEOM   = "geometry"
MALHA_CAT    = "categoria_fundiaria_v2025"

# Amazonia Legal (IBGE 2022)
AMZ_SCHEMA = "ibge"
AMZ_TABLE  = "pa_br_amazonialegal_ibge_2022"
AMZ_GEOM   = "geometry"

# Desmatamento PRODES (INPE)
DESMAT_SCHEMA = "inpe"
DESMAT_TABLE  = "pa_mt_desmatamento_inpe_20260304"
DESMAT_GEOM   = "geometry"
DESMAT_DATA   = "image_date"

# SIGEF / SNCI (apenas para visualizacao no mapa)
SIGEF_SCHEMA = "incra"; SIGEF_TABLE = "pa_br_sigef_privado_incra_2026"; SIGEF_GEOM = "geometry"
SNCI_SCHEMA  = "incra"; SNCI_TABLE  = "pa_br_snci_privado_incra_2026";  SNCI_GEOM  = "geometry"

# ============================================================================
# LIMIARES E DATAS DE REFERENCIA
# ============================================================================
DATA_CORTE_LEI    = date(2008, 7, 22)  # Lei 11.952/2009
LIMIAR_AMZ_LEGAL  = 50.0   # >50% da area do imovel sobreposta -> dentro da Amazonia Legal
LIMIAR_PREDOM     = 50.0   # >50% da area do imovel em uma categoria -> categoria predominante
AREA_LIMITE_HA    = 2500.0 # ha — limite da Lei 11.952/2009

print(f"Imovel selecionado: {COD_IMOVEL}")


Imovel selecionado: MT-5106265-62DB7655979E41A5BB9FA3A0CD2AF9BD


In [4]:
# Conexao com o banco
encoded_pw = urllib.parse.quote_plus(DB_PASSWORD)
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{encoded_pw}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print(f"[OK] Conectado ao banco {DB_NAME} em {DB_HOST}:{DB_PORT}")
except Exception as e:
    print(f"[ERRO] Falha na conexao: {e}")


[OK] Conectado ao banco geotec em localhost:5432


<hr style="border:1px solid #2c6e9e;">

## 3. Informações do Imóvel (CAR)

Consulta a área (ha) e o número de módulos fiscais do imóvel — variáveis necessárias para a hierarquia de mensagens da Lei 11.952/2009.


In [5]:
query_car = f"""
SELECT
    c.{CAR_COD}                                                           AS cod_imovel,
    c.nom_tema                                                            AS nome,
    c.municipio,
    c.cod_estado                                                          AS uf,
    ROUND(c.num_area::numeric, 4)                                         AS area_declarada_ha,
    ROUND(ST_Area(ST_Transform(c.{CAR_GEOM}, 97823))::numeric / 10000, 4) AS area_calculada_ha,
    ROUND(
        (ST_Area(ST_Transform(c.{CAR_GEOM}, 97823)) / 10000 / NULLIF(mf.modulofiscal_ha, 0))
    ::numeric, 2)                                                         AS num_mf_incra
FROM {CAR_SCHEMA}.{CAR_TABLE} c
LEFT JOIN incra.pa_br_modulosfiscais_incra mf
       ON SPLIT_PART(c.{CAR_COD}, '-', 2)::integer = mf.codibge
WHERE c.{CAR_COD} = '{COD_IMOVEL}'
"""

df_car = pd.read_sql(text(query_car), con=engine)

if df_car.empty:
    raise RuntimeError(f"Imovel nao encontrado: {COD_IMOVEL}")

area_ha      = float(df_car["area_calculada_ha"].iloc[0])
num_mf_incra = float(df_car["num_mf_incra"].iloc[0] or 0)

display(HTML(f"""
<h3>Imovel analisado: {df_car['cod_imovel'].iloc[0]}</h3>
<table border='1' cellpadding='6' style='border-collapse:collapse; font-size:0.95em'>
  <tr><td><b>Codigo</b></td><td>{df_car['cod_imovel'].iloc[0]}</td></tr>
  <tr><td><b>Municipio / UF</b></td><td>{df_car['municipio'].iloc[0]} / {df_car['uf'].iloc[0]}</td></tr>
  <tr><td><b>Area declarada</b></td><td>{df_car['area_declarada_ha'].iloc[0]:,.2f} ha</td></tr>
  <tr><td><b>Area calculada</b></td><td>{area_ha:,.2f} ha</td></tr>
  <tr><td><b>N. de Modulos Fiscais (INCRA)</b></td><td>{num_mf_incra:,.2f} MF</td></tr>
  <tr><td><b>Limite Lei 11.952/2009</b></td><td>{AREA_LIMITE_HA:,.0f} ha (imovel {"acima" if area_ha > AREA_LIMITE_HA else "abaixo"} do limite)</td></tr>
</table>
"""))


Codigo,MT-5106265-62DB7655979E41A5BB9FA3A0CD2AF9BD
Municipio / UF,Novo Mundo / MT
Area declarada,"8,556.25 ha"
Area calculada,"8,555.41 ha"
N. de Modulos Fiscais (INCRA),95.06 MF
Limite Lei 11.952/2009,"2,500 ha (imovel acima do limite)"


<hr style="border:1px solid #2c6e9e;">

## 4. Categoria Fundiária Predominante

Identifica qual categoria da **Malha Fundiária CDT 2025** ocupa a maior parcela do imóvel. Para a Lei 11.952/2009, interessa especificamente quando a categoria predominante (>50% da área) for **GP** ou **GPFPND**.


In [6]:
query_predom = f"""
WITH car AS (
    SELECT {CAR_GEOM} AS geom,
           ST_Area(ST_Transform({CAR_GEOM}, 97823)) / 10000 AS area_ha
    FROM {CAR_SCHEMA}.{CAR_TABLE}
    WHERE {CAR_COD} = '{COD_IMOVEL}'
),
inter AS (
    SELECT
        l.{MALHA_CAT} AS categoria,
        ST_Area(ST_Transform(
            ST_CollectionExtract(ST_MakeValid(ST_Intersection(c.geom, l.{MALHA_GEOM})), 3),
            97823
        )) / 10000 AS area_ha
    FROM {MALHA_SCHEMA}.{MALHA_TABLE} l
    CROSS JOIN car c
    WHERE ST_Intersects(c.geom, l.{MALHA_GEOM})
)
SELECT
    i.categoria,
    ROUND(SUM(i.area_ha)::numeric, 4) AS area_ha,
    ROUND((SUM(i.area_ha) / NULLIF(MAX(c.area_ha), 0) * 100)::numeric, 2) AS perc
FROM inter i CROSS JOIN car c
WHERE i.area_ha > 0
GROUP BY i.categoria
ORDER BY area_ha DESC
"""

df_predom = pd.read_sql(text(query_predom), con=engine)

cat_predominante = None
perc_predominante = 0.0
if not df_predom.empty and df_predom["perc"].iloc[0] > LIMIAR_PREDOM:
    cat_predominante = df_predom["categoria"].iloc[0]
    perc_predominante = float(df_predom["perc"].iloc[0])

display(HTML("<b>Sobreposicoes com a Malha Fundiaria CDT 2025:</b>"))
display(df_predom)

if cat_predominante:
    print(f"\n[OK] Categoria predominante: {cat_predominante} ({perc_predominante:.1f}% do imovel)")
else:
    print("\n[INFO] Nenhuma categoria da malha cobre mais de 50% do imovel — sem predominancia identificada.")


,categoria,area_ha,perc
0,GPFPND,8144.2085,95.1900
1,GP,406.7395,4.7500
2,ASRFG,4.4661,0.0500



[OK] Categoria predominante: GPFPND (95.2% do imovel)


<hr style="border:1px solid #2c6e9e;">

## 5. Localização em Relação à Amazônia Legal

Calcula a porcentagem da área do imóvel que está dentro do limite oficial da **Amazônia Legal** (IBGE 2022).
Considera-se que o imóvel está **dentro da Amazônia Legal** quando essa porcentagem é superior a **50%** — critério usado pela Lei 11.952/2009.


In [7]:
query_amz = f"""
WITH car AS (
    SELECT {CAR_GEOM} AS geom,
           ST_Area(ST_Transform({CAR_GEOM}, 97823)) / 10000 AS area_ha
    FROM {CAR_SCHEMA}.{CAR_TABLE}
    WHERE {CAR_COD} = '{COD_IMOVEL}'
)
SELECT
    ROUND((ST_Area(ST_Transform(ST_Intersection(c.geom, a.{AMZ_GEOM}), 97823)) / 10000)::numeric, 4) AS area_sobrep_ha,
    ROUND(((ST_Area(ST_Transform(ST_Intersection(c.geom, a.{AMZ_GEOM}), 97823)) / 10000) / NULLIF(c.area_ha, 0) * 100)::numeric, 2) AS perc
FROM {AMZ_SCHEMA}.{AMZ_TABLE} a
CROSS JOIN car c
WHERE ST_Intersects(c.geom, a.{AMZ_GEOM})
"""

df_amz = pd.read_sql(text(query_amz), con=engine)

if df_amz.empty:
    perc_amz = 0.0
    area_sobrep_amz = 0.0
else:
    perc_amz = float(df_amz["perc"].iloc[0] or 0)
    area_sobrep_amz = float(df_amz["area_sobrep_ha"].iloc[0] or 0)

dentro_amazonia_legal = perc_amz > LIMIAR_AMZ_LEGAL

cor_status = "#28a745" if dentro_amazonia_legal else "#6c757d"
status_txt = "DENTRO" if dentro_amazonia_legal else "FORA"

display(HTML(f"""
<table border='1' cellpadding='6' style='border-collapse:collapse; font-size:0.95em'>
  <tr><td><b>Area sobreposta a Amazonia Legal</b></td><td>{area_sobrep_amz:,.2f} ha</td></tr>
  <tr><td><b>% do imovel sobreposto</b></td><td>{perc_amz:.2f}%</td></tr>
  <tr><td><b>Limiar para dentro/fora</b></td><td>{LIMIAR_AMZ_LEGAL:.0f}%</td></tr>
  <tr><td><b>Status</b></td><td><span style='color:{cor_status};font-weight:bold'>{status_txt} da Amazonia Legal</span></td></tr>
</table>
"""))


Area sobreposta a Amazonia Legal,"8,555.41 ha"
% do imovel sobreposto,100.00%
Limiar para dentro/fora,50%
Status,DENTRO da Amazonia Legal


<hr style="border:1px solid #2c6e9e;">

## 6. Tipificação da Ocupação (PRODES — INPE)

Identifica a data de detecção mais antiga de **desmatamento PRODES** que intersecta o imóvel.
A Lei 11.952/2009 utiliza **22/07/2008** como data de corte:
- **Pré 22/07/2008** → ocupação anterior comprovável por sensoriamento remoto (favorece a regularização)
- **Pós 22/07/2008** → ocupação posterior, exige comprovação documental adicional
- **Sem desmatamento detectado** → não há evidência por sensoriamento remoto (tratado como pós 2008)


In [8]:
query_prodes = f"""
WITH car AS (
    SELECT {CAR_GEOM} AS geom
    FROM {CAR_SCHEMA}.{CAR_TABLE}
    WHERE {CAR_COD} = '{COD_IMOVEL}'
)
SELECT
    MIN(p.{DESMAT_DATA})  AS data_mais_antiga,
    MAX(p.{DESMAT_DATA})  AS data_mais_recente,
    COUNT(*)              AS n_poligonos
FROM {DESMAT_SCHEMA}.{DESMAT_TABLE} p
CROSS JOIN car c
WHERE ST_Intersects(p.{DESMAT_GEOM}, c.geom)
"""

df_prodes = pd.read_sql(text(query_prodes), con=engine)

n_poligonos = int(df_prodes["n_poligonos"].iloc[0] or 0)
data_mais_antiga = df_prodes["data_mais_antiga"].iloc[0]
data_mais_recente = df_prodes["data_mais_recente"].iloc[0]

# Normaliza para tipo date
if data_mais_antiga is not None and hasattr(data_mais_antiga, "date"):
    data_mais_antiga = data_mais_antiga.date()
if data_mais_recente is not None and hasattr(data_mais_recente, "date"):
    data_mais_recente = data_mais_recente.date()

eh_pre_2008 = (data_mais_antiga is not None) and (data_mais_antiga < DATA_CORTE_LEI)

if data_mais_antiga is None:
    status_prodes = "Sem desmatamento PRODES intersectando o imovel"
    cor_prodes = "#6c757d"
elif eh_pre_2008:
    status_prodes = f"Pre 22/07/2008 (data mais antiga: {data_mais_antiga.isoformat()})"
    cor_prodes = "#28a745"
else:
    status_prodes = f"Pos 22/07/2008 (data mais antiga: {data_mais_antiga.isoformat()})"
    cor_prodes = "#fd7e14"

display(HTML(f"""
<table border='1' cellpadding='6' style='border-collapse:collapse; font-size:0.95em'>
  <tr><td><b>Poligonos PRODES intersectando o imovel</b></td><td>{n_poligonos}</td></tr>
  <tr><td><b>Data mais antiga</b></td><td>{data_mais_antiga if data_mais_antiga else "—"}</td></tr>
  <tr><td><b>Data mais recente</b></td><td>{data_mais_recente if data_mais_recente else "—"}</td></tr>
  <tr><td><b>Data de corte (Lei 11.952/2009)</b></td><td>{DATA_CORTE_LEI.isoformat()}</td></tr>
  <tr><td><b>Tipificacao</b></td><td><span style='color:{cor_prodes};font-weight:bold'>{status_prodes}</span></td></tr>
</table>
"""))


Poligonos PRODES intersectando o imovel,7
Data mais antiga,2008-08-10
Data mais recente,2008-08-10
Data de corte (Lei 11.952/2009),2008-07-22
Tipificacao,Pos 22/07/2008 (data mais antiga: 2008-08-10)


<hr style="border:1px solid #2c6e9e;">

## 7. Mensagem do Indicador (Lei 11.952/2009)

Aplica a hierarquia de mensagens da Lei 11.952/2009 para o imóvel analisado, considerando:

| Condição | Mensagem |
|---|---|
| GP/GPFPND predominante + dentro Amaz. Legal + área > 2.500 ha | **NÃO regularizável** (excede limite de área) |
| GP/GPFPND predominante + dentro Amaz. Legal + área ≤ 2.500 ha + pós 2008 | **Possivelmente NÃO regularizável** (sem ocupação pré-2008 detectada) |
| GP/GPFPND predominante + dentro Amaz. Legal + área ≤ 2.500 ha + pré 2008 | **Possivelmente regularizável** (atende todos os critérios) |
| GP/GPFPND predominante + fora Amaz. Legal | Verificar documentação (Lei 11.952/2009 não se aplica diretamente) |
| Categoria predominante diferente de GP/GPFPND | Lei 11.952/2009 não se aplica |


In [9]:
mensagens = []
html_msgs = []

TITULO_LEI    = "REGULARIZACAO FUNDIARIA - Lei 11.952/2009"
TITULO_NA     = "LEI 11.952/2009 NAO SE APLICA"

CSS_VERDE     = "background:#d4edda;border-left:5px solid #28a745;padding:12px 15px;border-radius:4px;margin:8px 0;color:#0a3d1c"
CSS_AMARELO   = "background:#fff3cd;border-left:5px solid #ffc107;padding:12px 15px;border-radius:4px;margin:8px 0;color:#5c4400"
CSS_VERMELHO  = "background:#f8d7da;border-left:5px solid #dc3545;padding:12px 15px;border-radius:4px;margin:8px 0;color:#4a1f24"
CSS_NEUTRO    = "background:#e9ecef;border-left:5px solid #6c757d;padding:12px 15px;border-radius:4px;margin:8px 0;color:#212529"

def add_msg(titulo, msg, css):
    mensagens.append((titulo, msg))
    html_msgs.append(f"<div style='{css}'><b>{titulo}</b><br>{msg}</div>")

CATEGORIAS_PUBLICAS = {
    "GP":     "Gleba Publica (GP)",
    "GPFPND": "Floresta Publica do Tipo B (GPFPND)",
}

if cat_predominante in CATEGORIAS_PUBLICAS:
    nome_cat = CATEGORIAS_PUBLICAS[cat_predominante]

    if dentro_amazonia_legal:
        if area_ha > AREA_LIMITE_HA:
            # Mensagem E - >2500 ha → NAO regularizavel
            add_msg(TITULO_LEI,
                    f"Imovel predominantemente sobreposto a {nome_cat} ({perc_predominante:.1f}% da area). "
                    f"E importante verificar se o imovel possui titulo de propriedade. Caso nao possua, "
                    f"este imovel <b>NAO configura uma posse regularizavel</b> de acordo com a Lei 11.952/2009 "
                    f"por possuir mais de 2.500 hectares de area total ({area_ha:,.2f} ha).",
                    CSS_VERMELHO)
        elif eh_pre_2008:
            # Mensagem G - <=2500 + pre 2008 → possivelmente regularizavel
            add_msg(TITULO_LEI,
                    f"Imovel predominantemente sobreposto a {nome_cat} ({perc_predominante:.1f}% da area). "
                    f"E importante verificar se o imovel possui titulo de propriedade. Caso nao possua, "
                    f"este imovel <b>possivelmente CONFIGURA uma posse regularizavel</b> de acordo com a "
                    f"Lei 11.952/2009, por possuir ocupacao anterior a 22/07/2008 verificada por sensoriamento "
                    f"remoto (PRODES desde {data_mais_antiga.isoformat()}) e ter menos de 2.500 hectares "
                    f"de area total ({area_ha:,.2f} ha).",
                    CSS_VERDE)
        else:
            # Mensagem F - <=2500 + pos 2008 (ou sem PRODES) → possivelmente NAO regularizavel
            comp_remoto = ("nao possuir ocupacao anterior a 22/07/2008 verificavel por sensoriamento remoto"
                           if data_mais_antiga is not None
                           else "nao haver poligonos PRODES intersectando o imovel para comprovacao remota")
            add_msg(TITULO_LEI,
                    f"Imovel predominantemente sobreposto a {nome_cat} ({perc_predominante:.1f}% da area). "
                    f"E importante verificar se o imovel possui titulo de propriedade. Caso nao possua, "
                    f"este imovel <b>possivelmente NAO configura uma posse regularizavel</b> de acordo com a "
                    f"Lei 11.952/2009, por {comp_remoto}, salvo se o proprietario puder comprovar a ocupacao "
                    f"de forma documental.",
                    CSS_AMARELO)
    else:
        # Mensagem H - fora da Amazonia Legal
        add_msg(TITULO_LEI,
                f"Imovel predominantemente sobreposto a {nome_cat} ({perc_predominante:.1f}% da area), "
                f"localizado <b>fora da Amazonia Legal</b> ({perc_amz:.1f}% sobreposto ao limite). "
                f"E importante verificar se o imovel possui titulo de propriedade. Caso nao possua, "
                f"verificar a documentacao existente para qualificar a posse. A Lei 11.952/2009 nao se "
                f"aplica diretamente a este caso.",
                CSS_NEUTRO)
else:
    # Categoria nao e GP nem GPFPND
    if cat_predominante:
        nome_cat = cat_predominante
    else:
        nome_cat = "(sem predominancia identificada)"
    add_msg(TITULO_NA,
            f"Categoria fundiaria predominante: <b>{nome_cat}</b>. A Lei 11.952/2009 trata especificamente "
            f"de imoveis predominantemente sobrepostos a Gleba Publica (GP) ou Floresta Publica do Tipo B "
            f"(GPFPND). Este imovel nao se enquadra nas hipoteses da lei.",
            CSS_NEUTRO)

display(HTML("<h3>Output do indicador (Lei 11.952/2009)</h3>"
             "<p style='font-size:0.9em;color:#666'>Esta analise e <b>complementar</b> a do notebook "
             "<code>02_indicadores_fundiarios.ipynb</code> (CMN n. 5.268). Se houver impedimento ou "
             "observacao naquele notebook, esta mensagem deve ser interpretada em conjunto.</p>"
             + "".join(html_msgs)))


<hr style="border:1px solid #2c6e9e;">

## 8. Visualização — Imóvel, Amazônia Legal e PRODES

Mapa interativo com as mesmas facilidades visuais usadas no notebook `02_indicadores_fundiarios.ipynb`:
- **Botões de zoom por camada** para navegar rapidamente entre os temas carregados
- **Painel fixo de informações** no canto inferior direito ao passar o mouse sobre as feições
- **Realce visual no hover** para ajudar a identificar a geometria selecionada
- **Controle de camadas** para alternar entre satélite, mapa claro e temas analíticos

Camadas próprias da regularização fundiária:
- **CAR** (contorno preto) — imóvel analisado
- **Amazônia Legal** (linha azul) — limite IBGE 2022
- **GP** e **GPFPND** (vermelho) — categorias da Malha Fundiária
- **PRODES pré 22/07/2008** (verde) — evidência de ocupação anterior à data de corte
- **PRODES pós 22/07/2008** (marrom) — desmatamento posterior à data de corte
- **SIGEF/SNCI** (azul) — registros privados para contexto

In [ ]:
# ============================================================================
# Carrega geometrias para o mapa
# ============================================================================

def para_wgs84(gdf):
    if gdf is None or gdf.empty:
        return gpd.GeoDataFrame()
    if gdf.crs is None:
        return gdf.set_crs(epsg=4674).to_crs(epsg=4326)
    if gdf.crs.to_epsg() != 4326:
        return gdf.to_crs(epsg=4326)
    return gdf


def carregar_geom(query, geom_col):
    try:
        return gpd.read_postgis(query, con=engine, geom_col=geom_col)
    except Exception as e:
        print(f"  [AVISO] Não foi possível carregar: {e}")
        return gpd.GeoDataFrame()


print("Carregando geometrias...")

# CAR
gdf_car = gpd.read_postgis(
    f"SELECT * FROM {CAR_SCHEMA}.{CAR_TABLE} WHERE {CAR_COD} = '{COD_IMOVEL}'",
    con=engine, geom_col=CAR_GEOM
)

# Amazônia Legal: recorte por bbox do CAR + buffer para manter o mapa leve.
gdf_amz = carregar_geom(f"""
    WITH car AS (
        SELECT ST_Buffer(ST_Envelope({CAR_GEOM}), 0.5) AS bbox
        FROM {CAR_SCHEMA}.{CAR_TABLE} WHERE {CAR_COD} = '{COD_IMOVEL}'
    )
    SELECT ST_Multi(ST_CollectionExtract(ST_Intersection(a.{AMZ_GEOM}, c.bbox), 3)) AS {AMZ_GEOM}
    FROM {AMZ_SCHEMA}.{AMZ_TABLE} a CROSS JOIN car c
    WHERE ST_Intersects(a.{AMZ_GEOM}, c.bbox)
""", geom_col=AMZ_GEOM)

# GP e GPFPND consolidados que intersectam o imóvel.
def carregar_malha_consolidada(categoria):
    return carregar_geom(f"""
        WITH car AS (
            SELECT {CAR_GEOM} AS geom FROM {CAR_SCHEMA}.{CAR_TABLE} WHERE {CAR_COD} = '{COD_IMOVEL}'
        ), consolidado AS (
            SELECT ST_Multi(ST_CollectionExtract(ST_UnaryUnion(ST_Collect(ST_Intersection(c.geom, l.{MALHA_GEOM}))), 3)) AS {MALHA_GEOM}
            FROM {MALHA_SCHEMA}.{MALHA_TABLE} l JOIN car c ON ST_Intersects(l.{MALHA_GEOM}, c.geom)
            WHERE l.{MALHA_CAT} = '{categoria}'
        )
        SELECT '{categoria}' AS categoria, {MALHA_GEOM}
        FROM consolidado WHERE {MALHA_GEOM} IS NOT NULL AND NOT ST_IsEmpty({MALHA_GEOM})
    """, geom_col=MALHA_GEOM)


gdf_gp     = carregar_malha_consolidada("GP")
gdf_gpfpnd = carregar_malha_consolidada("GPFPND")

# PRODES intersectando o imóvel, separado por período.
gdf_prodes_pre = carregar_geom(f"""
    WITH car AS (
        SELECT {CAR_GEOM} AS geom FROM {CAR_SCHEMA}.{CAR_TABLE} WHERE {CAR_COD} = '{COD_IMOVEL}'
    )
    SELECT p.id, p.{DESMAT_DATA}, p.year, p.class_name, p.{DESMAT_GEOM}
    FROM {DESMAT_SCHEMA}.{DESMAT_TABLE} p JOIN car c ON ST_Intersects(p.{DESMAT_GEOM}, c.geom)
    WHERE p.{DESMAT_DATA} < '{DATA_CORTE_LEI.isoformat()}'
""", geom_col=DESMAT_GEOM)

gdf_prodes_pos = carregar_geom(f"""
    WITH car AS (
        SELECT {CAR_GEOM} AS geom FROM {CAR_SCHEMA}.{CAR_TABLE} WHERE {CAR_COD} = '{COD_IMOVEL}'
    )
    SELECT p.id, p.{DESMAT_DATA}, p.year, p.class_name, p.{DESMAT_GEOM}
    FROM {DESMAT_SCHEMA}.{DESMAT_TABLE} p JOIN car c ON ST_Intersects(p.{DESMAT_GEOM}, c.geom)
    WHERE p.{DESMAT_DATA} >= '{DATA_CORTE_LEI.isoformat()}'
""", geom_col=DESMAT_GEOM)

# SIGEF e SNCI entram como contexto registral no mapa.
gdf_sigef = carregar_geom(f"""
    WITH car AS (
        SELECT {CAR_GEOM} AS geom FROM {CAR_SCHEMA}.{CAR_TABLE} WHERE {CAR_COD} = '{COD_IMOVEL}'
    )
    SELECT l.* FROM {SIGEF_SCHEMA}.{SIGEF_TABLE} l JOIN car c ON ST_Intersects(l.{SIGEF_GEOM}, c.geom)
""", geom_col=SIGEF_GEOM)

gdf_snci = carregar_geom(f"""
    WITH car AS (
        SELECT {CAR_GEOM} AS geom FROM {CAR_SCHEMA}.{CAR_TABLE} WHERE {CAR_COD} = '{COD_IMOVEL}'
    )
    SELECT l.* FROM {SNCI_SCHEMA}.{SNCI_TABLE} l JOIN car c ON ST_Intersects(l.{SNCI_GEOM}, c.geom)
""", geom_col=SNCI_GEOM)

print("[OK] Geometrias carregadas")

# ============================================================================
# Monta o mapa
# ============================================================================

CORES = {
    "SIGEF":      ("#1a4a6b", 0.25, "SIGEF"),
    "SNCI":       ("#2e6da4", 0.20, "SNCI"),
    "GP":         ("#cd5c5c", 0.35, "GP"),
    "GPFPND":     ("#8b0000", 0.50, "GPFPND"),
    "PRODES_PRE": ("#28a745", 0.55, "PRODES pré 2008"),
    "PRODES_POS": ("#a52a2a", 0.55, "PRODES pós 2008"),
    "AMZ":        ("#1f77b4", 0.00, "Amazônia Legal"),
    "CAR":        ("#000000", 0.00, "CAR"),
}


def estilo_preenchido(cor, alpha, peso=1):
    return lambda _feature: {
        "fillColor": cor,
        "color": cor,
        "weight": peso,
        "fillOpacity": alpha,
    }


def estilo_linha(cor, peso=3):
    return lambda _feature: {
        "fillColor": cor,
        "color": cor,
        "weight": peso,
        "fillOpacity": 0,
    }


_tooltip_registry = {}


def adicionar_camada(mapa, gdf, chave_cor, nome_grupo, mostrar=False, peso=1, contorno=False,
                     tooltip_fields=None, tooltip_aliases=None):
    if gdf is None or gdf.empty:
        return None
    gdf_plot = para_wgs84(gdf).copy()

    # Folium/GeoJSON não serializa Timestamp automaticamente.
    for col in gdf_plot.columns:
        if col != gdf_plot.geometry.name and pd.api.types.is_datetime64_any_dtype(gdf_plot[col]):
            gdf_plot[col] = gdf_plot[col].astype(str)

    gdf_plot["_camada"] = nome_grupo

    grupo = folium.FeatureGroup(name=nome_grupo, show=mostrar)
    cor, alpha, _ = CORES[chave_cor]
    estilo = estilo_linha(cor, peso=peso) if contorno else estilo_preenchido(cor, alpha, peso=peso)
    campos_painel = ["_camada"]
    aliases_painel = ["Camada: "]
    if tooltip_fields:
        for campo, alias in zip(tooltip_fields, tooltip_aliases or tooltip_fields):
            if campo in gdf_plot.columns:
                campos_painel.append(campo)
                aliases_painel.append(alias)

    geojson_obj = folium.GeoJson(
        gdf_plot.to_json(),
        style_function=estilo,
        highlight_function=lambda _feature: {
            "weight": max(peso + 1, 3),
            "fillOpacity": min(alpha + 0.15, 0.8),
            "color": cor,
        },
    )
    geojson_obj.add_to(grupo)
    _tooltip_registry[geojson_obj.get_name()] = {"campos": campos_painel, "aliases": aliases_painel}
    grupo.add_to(mapa)
    return grupo


if gdf_car.empty:
    print("[ATENÇÃO] Geometria do CAR não disponível para montar o mapa.")
else:
    gdf_car_wgs84 = para_wgs84(gdf_car)
    car_3857 = gdf_car_wgs84.to_crs(epsg=3857)
    minx, miny, maxx, maxy = car_3857.total_bounds
    largura = maxx - minx
    altura = maxy - miny
    buffer_m = max(1000, max(largura, altura) * 0.40)
    area_foco = car_3857.buffer(buffer_m).union_all().envelope
    area_foco_wgs84 = gpd.GeoSeries([area_foco], crs=3857).to_crs(epsg=4326)
    foco_minx, foco_miny, foco_maxx, foco_maxy = area_foco_wgs84.total_bounds

    centro = [
        float((foco_miny + foco_maxy) / 2),
        float((foco_minx + foco_maxx) / 2),
    ]

    mapa = folium.Map(location=centro, zoom_start=12, tiles=None, control_scale=True, width="690px", height="440px")

    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Esri World Imagery",
        name="Satélite",
        overlay=False,
        control=True,
        show=True,
        max_zoom=19,
        max_native_zoom=19,
    ).add_to(mapa)
    folium.TileLayer(
        tiles="CartoDB positron",
        name="Mapa claro",
        overlay=False,
        control=True,
        show=False,
    ).add_to(mapa)

    camadas_zoom = {}

    camada = adicionar_camada(mapa, gdf_sigef, "SIGEF", CORES["SIGEF"][2], mostrar=False,
                              tooltip_fields=["nome_area", "status"],
                              tooltip_aliases=["Nome da área: ", "Status: "])
    if camada is not None: camadas_zoom["SIGEF"] = camada

    camada = adicionar_camada(mapa, gdf_snci, "SNCI", CORES["SNCI"][2], mostrar=False,
                              tooltip_fields=["nome_imove", "num_certif"],
                              tooltip_aliases=["Nome do imóvel: ", "Certificado: "])
    if camada is not None: camadas_zoom["SNCI"] = camada

    camada = adicionar_camada(mapa, gdf_gp, "GP", CORES["GP"][2], mostrar=True,
                              tooltip_fields=["categoria"],
                              tooltip_aliases=["Categoria: "])
    if camada is not None: camadas_zoom["GP"] = camada

    camada = adicionar_camada(mapa, gdf_gpfpnd, "GPFPND", CORES["GPFPND"][2], mostrar=True,
                              tooltip_fields=["categoria"],
                              tooltip_aliases=["Categoria: "])
    if camada is not None: camadas_zoom["GPFPND"] = camada

    camada = adicionar_camada(mapa, gdf_prodes_pre, "PRODES_PRE", CORES["PRODES_PRE"][2], mostrar=True,
                              tooltip_fields=[DESMAT_DATA, "year", "class_name"],
                              tooltip_aliases=["Data: ", "Ano: ", "Classe: "])
    if camada is not None: camadas_zoom["PRODES pré 2008"] = camada

    camada = adicionar_camada(mapa, gdf_prodes_pos, "PRODES_POS", CORES["PRODES_POS"][2], mostrar=True,
                              tooltip_fields=[DESMAT_DATA, "year", "class_name"],
                              tooltip_aliases=["Data: ", "Ano: ", "Classe: "])
    if camada is not None: camadas_zoom["PRODES pós 2008"] = camada

    camada = adicionar_camada(mapa, gdf_amz, "AMZ", CORES["AMZ"][2], mostrar=True, peso=2, contorno=True)
    if camada is not None: camadas_zoom["Amazônia Legal"] = camada

    camada = adicionar_camada(mapa, gdf_car, "CAR", CORES["CAR"][2], mostrar=True, peso=3, contorno=True)
    if camada is not None: camadas_zoom["CAR"] = camada

    folium.LayerControl(collapsed=True).add_to(mapa)
    mapa.fit_bounds([[foco_miny, foco_minx], [foco_maxy, foco_maxx]])

    # Painel de informações fixo no canto inferior direito.
    import json as _json_mod
    _map_var = mapa.get_name()
    _config_js = _json_mod.dumps(_tooltip_registry, ensure_ascii=False)
    _info_js = f"""
<script>
(function() {{
  var _tryCount = 0;
  function _initPanel() {{
    var mapObj = window['{_map_var}'];
    if (!mapObj) {{ if (_tryCount++ < 20) setTimeout(_initPanel, 150); return; }}
    var InfoPanel = L.Control.extend({{
      onAdd: function() {{
        this._div = L.DomUtil.create('div');
        this._div.style.cssText = (
          'background:white;padding:10px 14px;border-radius:6px;' +
          'border:1px solid #ccc;font-size:12px;font-family:Arial;' +
          'min-width:210px;max-width:300px;line-height:1.6;' +
          'box-shadow:2px 2px 8px rgba(0,0,0,0.15);pointer-events:none'
        );
        this._div.innerHTML = '<span style="color:#aaa;font-style:italic">Passe o mouse sobre uma feição</span>';
        return this._div;
      }}
    }});
    var panel = new InfoPanel({{position: 'bottomright'}});
    panel.addTo(mapObj);
    var panelDiv = panel._div;
    var config = {_config_js};
    Object.keys(config).forEach(function(lname) {{
      var layer = window[lname];
      if (!layer) return;
      layer.eachLayer(function(feat) {{
        feat.on('mouseover', function(e) {{
          var props = e.target.feature.properties;
          var c = config[lname];
          var html = '<table style="border-collapse:collapse">';
          c.campos.forEach(function(campo, i) {{
            var val = (props[campo] != null && props[campo] !== '') ? props[campo] : '—';
            html += '<tr><td style="font-weight:600;padding:1px 10px 1px 0;white-space:nowrap;color:#555">'
                  + c.aliases[i] + '</td><td style="color:#222">' + val + '</td></tr>';
          }});
          html += '</table>';
          panelDiv.innerHTML = html;
        }});
        feat.on('mouseout', function() {{
          panelDiv.innerHTML = '<span style="color:#aaa;font-style:italic">Passe o mouse sobre uma feição</span>';
        }});
      }});
    }});
  }}
  _initPanel();
}})();
</script>"""
    mapa.get_root().html.add_child(folium.Element(_info_js))

    botoes_zoom = "".join([
        f"<button onclick=\"zoomParaCamada_{mapa.get_name()}('{nome}')\" style='margin:2px;padding:2px 6px;font-size:11px;border:1px solid #bbb;border-radius:4px;background:#fff;cursor:pointer;'>{nome}</button>"
        for nome in camadas_zoom.keys()
    ])
    titulo_html = f"""
    <div style='position: fixed; top: 10px; left: 50px; z-index: 9999; background: rgba(255,255,255,0.88); padding: 6px 10px; border: 1px solid #d0d0d0; border-radius: 6px; font-size: 12px; max-width: 300px;'>
      <div style='line-height:1.6'><b>Zoom para camada:</b><br>{botoes_zoom}</div>
    </div>
    """
    linhas_camadas_js = ",\n".join([f"'{nome}': {grupo.get_name()}" for nome, grupo in camadas_zoom.items()])
    script_zoom = f"""
    <script>
    function zoomParaCamada_{mapa.get_name()}(nome) {{
      var mapa = {mapa.get_name()};
      var camadas = {{
        {linhas_camadas_js}
      }};
      var camada = camadas[nome];
      if (camada && camada.getBounds && camada.getBounds().isValid()) {{
        mapa.fitBounds(camada.getBounds(), {{padding: [20, 20]}});
      }}
    }}
    </script>
    """
    mapa.get_root().html.add_child(folium.Element(titulo_html))
    mapa.get_root().html.add_child(folium.Element(script_zoom))

    rotulo_imovel = COD_IMOVEL
    if "df_car" in globals() and not df_car.empty:
        nome_car = str(df_car["nome"].iloc[0]).strip()
        if nome_car and nome_car.lower() not in {"area do imovel", "área do imóvel"}:
            rotulo_imovel = nome_car
        else:
            rotulo_imovel = str(df_car["cod_imovel"].iloc[0]).strip()

    display(HTML(f"<h3>Mapa da regularização fundiária</h3><p><b>Imóvel analisado:</b> {rotulo_imovel}</p>"))
    display(mapa)

<hr style="border:1px solid #2c6e9e;">

## 9. Referências

- [Lei nº 11.952/2009 — Regularização Fundiária na Amazônia Legal](http://www.planalto.gov.br/ccivil_03/_Ato2007-2010/2009/Lei/L11952.htm)
- [PRODES — INPE / TerraBrasilis](http://terrabrasilis.dpi.inpe.br/app/dashboard/deforestation/biomes/legal_amazon/increments)
- [Limites da Amazônia Legal — IBGE](https://www.ibge.gov.br/geociencias/cartas-e-mapas/mapas-regionais/15819-amazonia-legal.html)
- [Cadastro Ambiental Rural (CAR) — SICAR](https://www.car.gov.br/)
- [Malha Fundiária — INCRA](https://www.gov.br/incra/pt-br/assuntos/governanca-fundiaria/malha-fundiaria)
- [Módulo Fiscal — INCRA](https://www.gov.br/incra/pt-br/assuntos/governanca-fundiaria/modulo-fiscal)
- [CMN nº 5.268 — Crédito Rural e Restrições Fundiárias](https://www.bcb.gov.br/)
